##### Module Imports

In [ ]:
import tools.eval as ev
import tools.serialTools as st
import tools.captureTools as ct
import pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

##### Global Vars

In [ ]:
datasetsPath = '../datasets/har/'

In [ ]:
xtrain =    pd.DataFrame()
xtest =     pd.DataFrame()
ytrain =    pd.DataFrame()
ytest =     pd.DataFrame()
bestIter = 0
initialAcc = 0

leGender = LabelEncoder()       # Female, Male
leHML = LabelEncoder()          # High, Moderate, Low
leChestPain = LabelEncoder()    # Non-anginal, Asymptomatic, Typical, Atypical
leThalassemia = LabelEncoder()  # Normal, Fixed Defect, Reversible Defect
leECG = LabelEncoder()          # Normal, ST-T abnormality, Left ventricular hypertrophy

##### Func: Import Data

In [ ]:
def importData():
    print(f'Importing data...')

    global xtrain, xtest, ytrain, ytest

    # Load dataset
    data = pd.read_csv(datasetsPath + 'heart_attack_risk_dataset.csv')

    # Encode Categorical Data
    data['Gender'] = leGender.fit_transform(data['Gender'])
    data['Physical_Activity_Level'] = leHML.fit_transform(data['Physical_Activity_Level'])
    data['Stress_Level'] = leHML.fit_transform(data['Stress_Level'])
    data['Heart_Attack_Risk'] = leHML.fit_transform(data['Heart_Attack_Risk'])
    data['Chest_Pain_Type'] = leChestPain.fit_transform(data['Chest_Pain_Type'])
    data['Thalassemia'] = leThalassemia.fit_transform(data['Thalassemia'])
    data['ECG_Results'] = leECG.fit_transform(data['ECG_Results'])

    # Splitting data into features and labels
    xdata = data.iloc[:,:19]
    ydata = data.iloc[:,19:]

    # Splitting data into training and testing
    xtrain, xtest, ytrain, ytest = train_test_split(
        xdata, 
        ydata, 
        test_size=0.2,
        random_state=0
    )

##### Func: Train Model

In [ ]:
def trainModel(model: XGBClassifier, feats: pd.DataFrame, labels: pd.DataFrame, setBestIter: bool = False, evalset: list = None):
    global bestIter
    
    if setBestIter == True:
        model.set_params(
            objective='multi:softmax',
            num_class=3,
            learning_rate=0.1,
            n_estimators=10000,
            early_stopping_rounds=100,
        )
        model.fit(
            feats, labels,
            eval_set = evalset,
            verbose = False
        )
        bestIter = model.best_iteration
    else:
        if bestIter == 0:
            print('BestIter = 0 -> Something is wrong!')
        model.set_params(
            objective='multi:softmax',
            num_class=3,
            learning_rate=0.1,
            n_estimators=bestIter,
            early_stopping_rounds=None,
        )
        model.fit(feats,labels)

##### Func: Train Quicksave

In [ ]:
def trainQuicksave():
    print("Training Quicksave...")
    model = XGBClassifier()
    evalset = [(xtrain,ytrain),(xtest,ytest)]
    trainModel(model, xtrain, ytrain, True, evalset)
    trainModel(model, xtrain, ytrain)
    model.save_model("quicksave.json")
    print("Quicksave model trained and saved as quicksave.json!")